In [1]:
# File paths
import os


project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
filename = "1997CanLII16226_ONCA"
anno = "llm"
version = "v4"
html_path = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}\{filename}_{anno}_{version}.html"
output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}"

os.makedirs(output_dir, exist_ok=True)

# Read HTML file
with open(html_path, 'r', encoding='utf-8') as file:
    html_content = file.read()
print(f"   ✓ HTML file loaded: {html_path}")


   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\1997CanLII16226_ONCA\1997CanLII16226_ONCA_llm_v4.html


In [2]:
# Import resolution utilities
import sys
sys.path.append(os.path.join(project_root, 'llm_based_annotation', 'resolution'))

from utils import (
    extract_spans_from_html, 
    filter_parent_spans,
    create_coresolution_clusters,
    print_clusters,
    print_clusters_simple,
    analyze_cluster_statistics,
    similarity_score
)

print("✓ Resolution utilities imported")

✓ Resolution utilities imported


In [3]:
# Extract all spans from HTML with context
print(f"\n{'='*80}")
print("EXTRACTING SPANS FROM HTML")
print(f"{'='*80}\n")

all_spans = extract_spans_from_html(html_path, label_type=["auto_label"],context_chars=200)
print(f"✓ Extracted {len(all_spans)} total spans")

# Display span distribution by label
from collections import Counter
label_counts = Counter([span.labelname for span in all_spans])
print(f"\nSpan distribution by label:")
for label, count in sorted(label_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {label}: {count}")


EXTRACTING SPANS FROM HTML

✓ Extracted 1792 total spans

Span distribution by label:
  title: 438
  reference: 391
  legislation: 361
  fragment: 347
  decision: 224
  secondary sources: 14
  source: 9
  authors: 8


In [7]:
# Filter to keep only parent-level labelnames (decision, legislation, secondary sources)
print(f"\n{'='*80}")
print("FILTERING TO PARENT LABELNAMES ONLY")
print(f"{'='*80}\n")

selected_labels = ['legislation', 'decision', 'secondary sources']

# Keep only spans where labelname is one of the parent categories
# This removes spans like 'title', 'reference', 'fragment', etc.
selected_spans = [span for span in all_spans 
                if span.labelname.lower() in [cat.lower() for cat in selected_labels]]

print(f"✓ Filtered from {len(all_spans)} to {len(selected_spans)} parent-level spans")

# Show distribution by labelname
parent_dist = Counter([span.labelname.lower() for span in selected_spans])

print(f"\nLabelname distribution:")
for category, count in sorted(parent_dist.items()):
    print(f"  {category}: {count}")


FILTERING TO PARENT LABELNAMES ONLY

✓ Filtered from 1792 to 599 parent-level spans

Labelname distribution:
  decision: 224
  legislation: 361
  secondary sources: 14


In [12]:
# Create coresolution clusters based on text similarity
print(f"\n{'='*80}")
print("CREATING CORESOLUTION CLUSTERS")
print(f"{'='*80}\n")

# Configuration
SIMILARITY_THRESHOLD = 0.6  # Adjust this (0-1): higher = stricter matching
SIMILARITY_METHOD = 'lcs'    # Options: 'lcs', 'substring', 'ratio', 'token'
GROUP_BY_PARENT = True       # Cluster only within same parent category

print(f"Configuration:")
print(f"  Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"  Similarity method: {SIMILARITY_METHOD}")
print(f"  Group by parent: {GROUP_BY_PARENT}")

# Create clusters
clusters = create_coresolution_clusters(
    selected_spans,
    similarity_threshold=SIMILARITY_THRESHOLD,
    similarity_method=SIMILARITY_METHOD,
    group_by_parent=GROUP_BY_PARENT
)

print(f"\n✓ Created {len(clusters)} clusters")

# Show statistics
stats = analyze_cluster_statistics(clusters)
print(f"\nCluster statistics:")
print(f"  Total spans clustered: {stats['total_spans']}")
print(f"  Average cluster size: {stats['avg_cluster_size']:.2f}")
print(f"  Max cluster size: {stats['max_cluster_size']}")
print(f"  Min cluster size: {stats['min_cluster_size']}")
print(f"  Singleton clusters: {stats['singleton_clusters']}")
print(f"  Multi-span clusters: {stats['multi_span_clusters']}")


CREATING CORESOLUTION CLUSTERS

Configuration:
  Similarity threshold: 0.6
  Similarity method: lcs
  Group by parent: True

✓ Created 172 clusters

Cluster statistics:
  Total spans clustered: 599
  Average cluster size: 3.48
  Max cluster size: 68
  Min cluster size: 1
  Singleton clusters: 87
  Multi-span clusters: 85


In [9]:
# Display clusters - Simple format (cluster ID + list of spans)
print(f"\n{'='*80}")
print("CORESOLUTION CLUSTERS - Simple View")
print(f"{'='*80}\n")

# Option 1: Show only multi-span clusters (more than 1 span)
multi_span_clusters = {cid: spans for cid, spans in clusters.items() if len(spans) > 1}
print(f"Multi-span clusters: {len(multi_span_clusters)} out of {len(clusters)} total\n")

if multi_span_clusters:
    print_clusters_simple(multi_span_clusters, max_text_len=80)
else:
    print("No multi-span clusters found. Try lowering the similarity threshold.")

# Option 2: To show ALL clusters (including singletons), uncomment below:
# print_clusters_simple(clusters, max_text_len=80)


CORESOLUTION CLUSTERS - Simple View

Multi-span clusters: 85 out of 172 total


Coresolution Clusters: 85 total

Cluster 0: [7 spans]
  - Regina v. Church of Scientology et al.
  - Indexed as: R. v. Church of Scientology
  - R. v. Church of Scientology (1992), 9 C.R.R
  - R. v. Church of Scientology
  - R. v. Church of Scientology; R. v. Zaharia
  - R. v. Church of Scientology; R. v. Zaharia
  - Church of Scientology of Toronto

Cluster 1: [2 spans]
  - 33 O.R. (3d) 65
  - 29 O.R. (3d) 320n

Cluster 3: [30 spans]
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - Charter
  - the Charter
  - Charter
  - Charter
  - Charter
  - Charter

Cluster 4: [9 spans]
  - Canadian Charter of Rights and Freedoms, ss. 8, 24(2)
  - Canadian Charter of Rights and F

## Cluster Analysis & Export

Now we have:
1. **`all_spans`**: All extracted spans with context
2. **`parent_spans`**: Filtered spans (only legislation, decision, secondary sources)
3. **`clusters`**: Dictionary mapping cluster_id → list of similar spans

You can:
- Adjust `SIMILARITY_THRESHOLD` (0-1) to control clustering strictness
- Change `SIMILARITY_METHOD`:
  - `'lcs'`: Longest Common Subsequence (flexible, allows gaps)
  - `'substring'`: Longest Common Substring (stricter, no gaps)
  - `'ratio'`: SequenceMatcher ratio (order-sensitive)
  - `'token'`: Token-based Jaccard similarity
- Export clusters for further processing

In [11]:
# Example: Export clusters to JSON
import json

def export_clusters_to_json(clusters, output_path):
    """Export clusters with span information to JSON."""
    export_data = {
        'metadata': {
            'total_clusters': len(clusters),
            'total_spans': sum(len(spans) for spans in clusters.values()),
            'similarity_threshold': SIMILARITY_THRESHOLD,
            'similarity_method': SIMILARITY_METHOD,
            'source_file': os.path.basename(html_path)
        },
        'clusters': {}
    }
    
    for cluster_id, spans in clusters.items():
        export_data['clusters'][str(cluster_id)] = [
            {
                'text': span.text,
                'labelname': span.labelname,
                'parent': span.parent,
                'top_level_parent': span.get_top_level_parent() if hasattr(span, 'get_top_level_parent') else '',
                'attributes': span.attributes,
                'start': span.start,
                'end': span.end
            }
            for span in spans
        ]
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Clusters exported to: {output_path}")

# Export clusters
clusters_output_path = os.path.join(output_dir, f"{filename}_coresolution_clusters.json")
export_clusters_to_json(clusters, clusters_output_path)

✓ Clusters exported to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\1997CanLII16226_ONCA\1997CanLII16226_ONCA_coresolution_clusters.json


## Similarity Testing

Test different similarity methods to understand how they work:

In [10]:
# Example: Compare similarity scores between two texts
text1 = "Supreme Court of Canada [2019] SCC 65"
text2 = "Supreme Court of Canada, 2019 SCC 65"

print(f"Text 1: {text1}")
print(f"Text 2: {text2}\n")

methods = ['lcs', 'substring', 'ratio', 'token']
for method in methods:
    score = similarity_score(text1, text2, method=method)
    print(f"{method:12s}: {score:.4f} ({score*100:.1f}%)")

Text 1: Supreme Court of Canada [2019] SCC 65
Text 2: Supreme Court of Canada, 2019 SCC 65

lcs         : 0.9459 (94.6%)
substring   : 0.6389 (63.9%)
ratio       : 0.9589 (95.9%)
token       : 0.5556 (55.6%)
